# 🤖 ISOM 260: Build Your First AI Agent

**Session 4, Meeting 1 — Your First Agent (Wed Sep 30)** | Suffolk University | Prof. Hasan Arslan

---

## What's an AI Agent?

In Session 3 your code made **API calls** and you CRAFTed prompts. But every call was one-shot: you ask, the model answers, and *you* decided every step. Today the model starts deciding for itself.

An **AI Agent** is different. It's an AI that can:

1. 🧠 **Think** about what it needs to do
2. 🛠️ **Use tools** — search the web, do math, look up data, take actions
3. 🔄 **Loop** — check its work, decide if it needs more info, and keep going
4. 💾 **Remember** — maintain context across steps

### The Formula

```
AI Agent = LLM + Tools + Reasoning Loop + Memory
```

### Why This Matters for Business

- **Chatbot**: "What's our revenue?" → "I don't have access to that data."
- **AI Agent**: "What's our revenue?" → *queries database* → *calculates growth* → *compares to forecast* → "Revenue is $2.3M, up 12% QoQ, 3% above forecast."

That's the leap. Let's build it.

---

### 💡 Real-World Context: NanoClaw & OpenClaw

Earlier this year (spring 2026), the hottest trend in AI was personal AI agents. **OpenClaw** — an open-source agent platform — exploded to 150,000+ users. But it has 400,000 lines of code and serious security issues (one user had their entire inbox deleted!).

Enter **NanoClaw** — built by a single developer with an AI coding agent in one weekend. Just **500 lines of core code**. Andrej Karpathy called it "really interesting" because it "fits into both my head and that of AI agents."

The lesson? **Simplicity wins.** Today, you'll build a mini-agent that captures the same core concept: an LLM that can use tools.

**Setup reminder:** your `GOOGLE_API_KEY` from Session 3 is already in Colab Secrets. `File → Save a copy in Drive`, then run top to bottom.


---

## 🚀 Part 1: Setup (2 minutes)

Install the Google Gemini SDK and set your API key.

In [ ]:
# Install the Google Gen AI Python SDK
!pip install -q google-genai

In [ ]:
# Set your API key
# Get yours at: https://aistudio.google.com/apikey
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the key icon in the left sidebar → Add secret named GOOGLE_API_KEY
try:
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    print("✅ API key loaded from Colab Secrets!")
except:
    # Option 2: Paste directly (less secure, but works for class)
    GOOGLE_API_KEY = "your-api-key-here"  # <-- Replace this
    print("⚠️ Using hardcoded API key. Consider using Colab Secrets instead.")

In [ ]:
# Verify connection
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL = "gemini-2.5-flash"      # one place to change the model for the whole notebook

response = client.models.generate_content(
    model=MODEL,
    contents="Say 'Agent ready!' in exactly 2 words."
)

print(response.text)
print(f"\n✅ Connection successful! Using model: {MODEL}")

---

## 🧠 Part 2: Understanding Tool Use (The Key Concept)

Here's the core idea that transforms a chatbot into an agent:

**Without tools:**
```
You: "What's 7,394 × 8,261?"
Gemini: "Let me calculate... approximately 61,097,834" (might be wrong!)
```

**With tools:**
```
You: "What's 7,394 × 8,261?"
Gemini: *thinks* "I should use the calculator tool for precision"
        *calls calculator(7394, 8261, 'multiply')*
        *gets back 61,081,834*
Gemini: "7,394 × 8,261 = 61,081,834" (✅ exact!)
```

### How It Works (The Agent Loop)

```
1. You send a message + tool definitions to Gemini
2. Gemini THINKS about whether it needs a tool
3. If yes: Gemini says "I want to call [tool] with [inputs]"
   → Your code EXECUTES the tool
   → You send the result BACK to Gemini
   → Gemini uses the result to form its answer
4. If no: Gemini just answers directly
```

This is the **exact same pattern** that powers NanoClaw, OpenClaw, Claude Code, Gemini's own agents, and every AI agent in the world. Let's build it!

---

## 🛠️ Part 3: Your First Tool — The Calculator

Let's give Gemini access to a calculator. We need:
1. A **tool definition** (tells Gemini what the tool does)
2. A **tool function** (the actual Python code that runs)
3. An **agent loop** (handles the back-and-forth)

In [ ]:
# ============================================
# STEP 1: Define the tool for Gemini
# ============================================
# This is like a job description — it tells Gemini what the tool can do

from google.genai import types

calculator_tool = types.FunctionDeclaration(
    name="calculator",
    description=(
        "A precise mathematical calculator. Use this tool whenever you need to "
        "perform arithmetic calculations. It handles addition, subtraction, "
        "multiplication, division, and exponentiation with perfect accuracy. "
        "Always prefer this tool over mental math for any calculation."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "The mathematical expression to evaluate, e.g. '2 + 2' or '(100 * 0.15) + 50'"
            }
        },
        "required": ["expression"]
    }
)

print("📝 Tool defined: calculator")
print(f"   Description: {calculator_tool.description[:80]}...")

In [ ]:
# ============================================
# STEP 2: Create the actual tool function
# ============================================
# This is the real Python code that runs when Gemini calls the tool

def calculator(expression: str) -> str:
    """
    Safely evaluate a mathematical expression.
    Returns the result as a string.
    """
    try:
        # Only allow safe math operations
        allowed_chars = set('0123456789+-*/.() ')
        if not all(c in allowed_chars for c in expression):
            return f"Error: Expression contains invalid characters. Only numbers and +-*/.() are allowed."

        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

# Test it!
print("Testing calculator:")
print(f"  2 + 2 = {calculator('2 + 2')}")
print(f"  7394 * 8261 = {calculator('7394 * 8261')}")
print(f"  (100 * 0.15) + 50 = {calculator('(100 * 0.15) + 50')}")
print("\n✅ Calculator tool is working!")

In [ ]:
# ============================================
# STEP 3: Build the Agent Loop!
# ============================================
# This is the HEART of every AI agent — the loop that connects
# Gemini's thinking to real-world tool execution

import json

def run_agent(user_message: str, tool_declarations: list, tool_functions: dict, verbose=True):
    """
    Run an AI agent that can use tools to answer questions.

    This is the same core pattern used by NanoClaw, OpenClaw,
    Claude Code, and every AI agent in production today.
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"🗣️  User: {user_message}")
        print(f"{'='*60}")

    # Build the conversation history
    contents = [
        types.Content(role="user", parts=[types.Part(text=user_message)])
    ]

    # Config with tools and system instruction
    config = types.GenerateContentConfig(
        tools=[types.Tool(function_declarations=tool_declarations)],
        system_instruction="You are a helpful assistant. Use the available tools whenever they would help you give a more accurate answer. Always show your reasoning."
    )

    step = 0

    while True:
        step += 1

        # 📡 Send message to Gemini (with tool definitions)
        response = client.models.generate_content(
            model=MODEL,
            config=config,
            contents=contents,
        )

        if verbose:
            print(f"\n🔄 Step {step}")

        # Get the model's response and add to history
        model_content = response.candidates[0].content
        contents.append(model_content)

        # Check if Gemini wants to use a tool
        function_calls = [p for p in model_content.parts if p.function_call]

        if not function_calls:
            # 🎉 No function calls — Gemini is done, extract final answer
            final_answer = response.text

            if verbose:
                print(f"\n🤖 Agent Answer:\n{final_answer}")
                print(f"\n✅ Done in {step} step(s)")

            return final_answer

        # Print any thinking text alongside function calls
        for part in model_content.parts:
            if not part.function_call and part.text:
                if verbose:
                    text = part.text
                    print(f"   🧠 Thinking: {text[:150]}..." if len(text) > 150 else f"   🧠 Thinking: {text}")

        # 🚀 Execute each function call
        function_response_parts = []
        for fc_part in function_calls:
            tool_name = fc_part.function_call.name
            tool_args = dict(fc_part.function_call.args)

            if verbose:
                print(f"   🛠️  Calling tool: {tool_name}")
                print(f"      Input: {json.dumps(tool_args, default=str)}")

            # Execute the tool!
            if tool_name in tool_functions:
                result = tool_functions[tool_name](**tool_args)
            else:
                result = f"Error: Unknown tool '{tool_name}'"

            if verbose:
                print(f"      Result: {result}")

            function_response_parts.append(
                types.Part.from_function_response(
                    name=tool_name,
                    response={"result": result}
                )
            )

        # Send function results back to Gemini
        contents.append(
            types.Content(role="user", parts=function_response_parts)
        )

        # Safety: prevent infinite loops
        if step > 10:
            return "Error: Agent exceeded maximum steps."

print("✅ Agent loop defined! Let's test it.")

In [ ]:
# ============================================
# 🎯 TEST IT! — Calculator Agent in Action
# ============================================

# Register our tools
tools = [calculator_tool]
tool_functions = {"calculator": calculator}

# Test 1: Simple math
run_agent("What is 7,394 multiplied by 8,261?", tools, tool_functions)

In [ ]:
# Test 2: Business calculation (needs multiple steps!)
run_agent(
    "I'm opening a coffee shop. My monthly rent is $4,500, ingredients cost $2,800, "
    "and staff costs $6,200. If I sell 3,400 cups per month at $5.75 each, "
    "what's my monthly profit? And how many months until I recoup a $45,000 startup investment?",
    tools, tool_functions
)

In [ ]:
# Test 3: Watch Gemini decide NOT to use the tool
run_agent(
    "What is the capital of France?",
    tools, tool_functions
)

### 🌟 Key Insight

Did you notice? Gemini **decided on its own** when to use the calculator and when not to. That's the magic — the LLM acts as the **brain** that decides which tools to use, when, and how. You don't write if/else logic. Gemini figures it out.

This is exactly what makes AI agents powerful: they **reason about** which actions to take.

---

## 📊 Part 4: Multi-Tool Agent (The Real Power)

One tool is cool. But real agents have **multiple tools** and decide which one(s) to use. Let's give our agent a toolkit:

1. 🧮 **Calculator** — math operations
2. 📈 **Stock Lookup** — get stock prices (simulated)
3. 🌐 **Company Info** — get company details (simulated)

With these three tools, Gemini can answer complex business questions like:
> "Compare the market caps of Apple and Microsoft and tell me which is larger"

In [ ]:
# ============================================
# Define multiple tools + their functions
# ============================================

# Tool 2: Stock price lookup (simulated data for classroom use)
stock_tool = types.FunctionDeclaration(
    name="get_stock_price",
    description=(
        "Look up the current stock price and key metrics for a publicly traded company. "
        "Provide the stock ticker symbol (e.g., AAPL for Apple, MSFT for Microsoft). "
        "Returns current price, daily change, 52-week high/low, and shares outstanding."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {
            "ticker": {
                "type": "string",
                "description": "Stock ticker symbol, e.g. 'AAPL', 'MSFT', 'GOOGL'"
            }
        },
        "required": ["ticker"]
    }
)

def get_stock_price(ticker: str) -> str:
    """Simulated stock data for classroom use."""
    stocks = {
        "AAPL": {"name": "Apple Inc.", "price": 242.58, "change": "+1.23%", "high_52w": 260.10, "low_52w": 164.08, "shares_outstanding": "15.12B"},
        "MSFT": {"name": "Microsoft Corp.", "price": 428.50, "change": "-0.45%", "high_52w": 468.35, "low_52w": 388.46, "shares_outstanding": "7.43B"},
        "GOOGL": {"name": "Alphabet Inc.", "price": 182.30, "change": "+0.87%", "high_52w": 201.42, "low_52w": 150.22, "shares_outstanding": "12.20B"},
        "AMZN": {"name": "Amazon.com Inc.", "price": 215.45, "change": "+2.10%", "high_52w": 242.52, "low_52w": 166.48, "shares_outstanding": "10.52B"},
        "TSLA": {"name": "Tesla Inc.", "price": 342.10, "change": "-3.21%", "high_52w": 488.54, "low_52w": 138.80, "shares_outstanding": "3.21B"},
        "NVDA": {"name": "NVIDIA Corp.", "price": 138.25, "change": "+1.95%", "high_52w": 153.13, "low_52w": 75.61, "shares_outstanding": "24.49B"},
    }
    ticker = ticker.upper().strip()
    if ticker in stocks:
        return json.dumps(stocks[ticker])
    return json.dumps({"error": f"Ticker '{ticker}' not found. Available: {', '.join(stocks.keys())}"})


# Tool 3: Company info
company_tool = types.FunctionDeclaration(
    name="get_company_info",
    description=(
        "Get detailed information about a company including its sector, "
        "number of employees, founding year, CEO, and headquarters location. "
        "Use the stock ticker symbol to look up the company."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {
            "ticker": {
                "type": "string",
                "description": "Stock ticker symbol, e.g. 'AAPL'"
            }
        },
        "required": ["ticker"]
    }
)

def get_company_info(ticker: str) -> str:
    """Simulated company info for classroom use."""
    companies = {
        "AAPL": {"name": "Apple Inc.", "sector": "Technology", "employees": 164000, "founded": 1976, "ceo": "Tim Cook", "hq": "Cupertino, CA"},
        "MSFT": {"name": "Microsoft Corp.", "sector": "Technology", "employees": 228000, "founded": 1975, "ceo": "Satya Nadella", "hq": "Redmond, WA"},
        "GOOGL": {"name": "Alphabet Inc.", "sector": "Technology", "employees": 182000, "founded": 1998, "ceo": "Sundar Pichai", "hq": "Mountain View, CA"},
        "AMZN": {"name": "Amazon.com Inc.", "sector": "Consumer Cyclical", "employees": 1540000, "founded": 1994, "ceo": "Andy Jassy", "hq": "Seattle, WA"},
        "TSLA": {"name": "Tesla Inc.", "sector": "Automotive", "employees": 140000, "founded": 2003, "ceo": "Elon Musk", "hq": "Austin, TX"},
        "NVDA": {"name": "NVIDIA Corp.", "sector": "Technology", "employees": 32000, "founded": 1993, "ceo": "Jensen Huang", "hq": "Santa Clara, CA"},
    }
    ticker = ticker.upper().strip()
    if ticker in companies:
        return json.dumps(companies[ticker])
    return json.dumps({"error": f"Company '{ticker}' not found."})


# Register all tools
all_tools = [calculator_tool, stock_tool, company_tool]
all_functions = {
    "calculator": calculator,
    "get_stock_price": get_stock_price,
    "get_company_info": get_company_info
}

print("✅ 3 tools registered:")
for t in all_tools:
    print(f"   🛠️  {t.name}")

In [ ]:
# 🎯 Multi-tool test: Gemini needs to use MULTIPLE tools and REASON across them

run_agent(
    "Compare Apple and Microsoft: which company has a higher market cap? "
    "Also tell me which company has more revenue per employee if Apple's "
    "annual revenue is $385 billion and Microsoft's is $245 billion.",
    all_tools, all_functions
)

In [ ]:
# 🎯 Another complex query

run_agent(
    "I have $100,000 to invest. If I split it equally between NVIDIA and Tesla, "
    "how many shares of each could I buy at current prices? "
    "Which company is older, and what sectors are they in?",
    all_tools, all_functions
)

### 🌟 Key Insight

Watch the step count! Gemini is making **multiple tool calls** in sequence, using the results from one to inform the next. It:

1. **Plans** what information it needs
2. **Calls** the right tools in the right order
3. **Synthesizes** the results into a coherent answer

That's an **agent**. Not a chatbot. An agent.

---

## 🏢 Part 5: Make It Enterprise-Grade (What Production Adds)

Your `run_agent` works. It would also get you fired in a real company. Four things every production agent has that yours doesn't:

| Problem | Toy agent | Production agent |
|---|---|---|
| The API rate-limits you (free tier: a few calls/minute) | crashes with `429` | **waits and retries** with backoff |
| A tool throws an error | crashes | **returns the error to the model**, which reports honestly |
| The model loops forever | burns money | a **circuit breaker** (max steps) |
| The agent wants to *send an email* / *issue a refund* | just does it | an **approval gate** — a human says yes |
| "What did the AI do last Tuesday?" | nobody knows | an **audit log** of every tool call |

That's `run_agent_v2`. Read it — every line maps to a row above.


In [ ]:
# ============================================
# run_agent_v2 — the production upgrades
# ============================================
import time, datetime
from google.genai import errors

AUDIT_LOG = []                 # every tool call, forever — this is what an auditor asks for
HUMAN_IN_THE_LOOP = False      # flip to True to approve risky actions live (you'll get an input() prompt)


def call_model_with_retry(contents, config, max_retries=4):
    """Free tiers rate-limit. Production code waits and retries instead of crashing."""
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except errors.APIError as e:
            if e.code in (429, 500, 503) and attempt < max_retries - 1:
                wait = 5 * (2 ** attempt)           # 5s → 10s → 20s (exponential backoff)
                print(f"   ⏳ API returned {e.code} — waiting {wait}s, retry {attempt + 1}/{max_retries - 1}")
                time.sleep(wait)
            else:
                raise


def approve(name, args):
    """The approval gate. In production this is a Slack button, a ticket, a manager's click."""
    print(f"   🚦 APPROVAL GATE: agent wants to run {name}({json.dumps(args, default=str)})")
    if not HUMAN_IN_THE_LOOP:
        print("      → HUMAN_IN_THE_LOOP is False, so this is auto-DENIED.")
        return False
    return input("      Approve? (y/n): ").strip().lower() == "y"


def run_agent_v2(user_message, tool_declarations, tool_functions,
                 risky_tools=(), max_steps=8, verbose=True):
    """Same loop as run_agent, plus retries, error handling, a circuit breaker,
    an approval gate for risky tools, and an audit log."""
    if verbose:
        print(f"\n{'=' * 60}\n🗣️  User: {user_message}\n{'=' * 60}")

    contents = [types.Content(role="user", parts=[types.Part(text=user_message)])]
    config = types.GenerateContentConfig(
        tools=[types.Tool(function_declarations=tool_declarations)],
        system_instruction=(
            "You are a careful business assistant. Use tools when they improve accuracy. "
            "If a tool returns an error or says it was BLOCKED, tell the user honestly and stop — "
            "never invent a result and never retry a blocked action."
        ),
    )

    for step in range(1, max_steps + 1):                       # ← circuit breaker
        response = call_model_with_retry(contents, config)     # ← retries
        model_content = response.candidates[0].content
        contents.append(model_content)
        parts = model_content.parts or []
        function_calls = [p for p in parts if p.function_call]

        if not function_calls:                                  # final answer
            answer = response.text or "(no answer returned)"
            if verbose:
                print(f"\n🤖 Agent Answer:\n{answer}\n\n✅ Done in {step} step(s)")
            return answer

        result_parts = []
        for part in function_calls:
            name, args = part.function_call.name, dict(part.function_call.args)
            t0 = time.time()
            if name not in tool_functions:
                result = f"Error: unknown tool '{name}'"
            elif name in risky_tools and not approve(name, args):    # ← approval gate
                result = "BLOCKED: this action requires human approval and it was not granted. Report this to the user."
            else:
                try:
                    result = tool_functions[name](**args)
                except Exception as e:                            # ← tool errors go back to the model
                    result = f"Error: tool '{name}' failed: {e}"

            AUDIT_LOG.append({                                    # ← audit log
                "time": datetime.datetime.now().strftime("%H:%M:%S"), "step": step,
                "tool": name, "args": json.dumps(args, default=str),
                "result": str(result)[:70], "ms": round((time.time() - t0) * 1000),
            })
            if verbose:
                print(f"   🛠️  {name}({json.dumps(args, default=str)}) → {str(result)[:100]}")
            result_parts.append(types.Part.from_function_response(name=name, response={"result": result}))

        contents.append(types.Content(role="user", parts=result_parts))

    return f"⚠️ Circuit breaker: stopped after {max_steps} steps without a final answer."


print("✅ run_agent_v2 ready: retries · error handling · circuit breaker · approval gate · audit log")


### A risky tool: `send_email`

Reading data is safe. **Acting** is where agents get dangerous. We add one simulated action tool and put it behind the gate.


In [ ]:
# A tool that ACTS in the world (simulated — nothing is really sent)
email_tool = types.FunctionDeclaration(
    name="send_email",
    description="Send an email on the user's behalf. Use ONLY when the user explicitly asks to send or email something.",
    parameters_json_schema={
        "type": "object",
        "properties": {
            "to":      {"type": "string", "description": "Recipient email address"},
            "subject": {"type": "string", "description": "Subject line"},
            "body":    {"type": "string", "description": "Email body, plain text"},
        },
        "required": ["to", "subject", "body"],
    },
)

def send_email(to: str, subject: str, body: str) -> str:
    return f"SENT (simulated) to {to} · subject '{subject}' · {len(body)} characters"

v2_tools = all_tools + [email_tool]
v2_functions = {**all_functions, "send_email": send_email}

# The gate is CLOSED (HUMAN_IN_THE_LOOP = False). Watch what the agent does when it's blocked.
run_agent_v2(
    "Compare Apple's and Microsoft's market caps (price × shares outstanding), "
    "then email a two-sentence summary to cfo@example.com with the subject 'Market cap check'.",
    v2_tools, v2_functions, risky_tools={"send_email"},
)


**Read the transcript.** The agent did the safe work (lookups, math), hit the gate on the risky action, and *reported* that it was blocked instead of pretending it sent the email. That honesty came from two lines: the `BLOCKED` result and the system instruction.

🎮 **YOUR TURN:** set `HUMAN_IN_THE_LOOP = True` in the cell below, re-run the query, and approve it yourself. *You* just became the control in the system. (Leave it `False` when you use "Run all" — `input()` blocks the notebook.)


In [ ]:
# HUMAN_IN_THE_LOOP = True     # ← uncomment, run, then re-run the query above and type y

# The audit trail — every tool call this session, with timing:
import pandas as pd
pd.DataFrame(AUDIT_LOG)


### 💼 Why a business student should care

- **Rate limits and retries** are the difference between a demo and an SLA.
- **Error propagation** ("tell the user the tool failed") is the #1 cure for confident hallucination inside agents.
- **The approval gate** is how you deploy an agent that can *act* without giving it a blank check. Regulators will ask for it. (Session 10 is entirely about this.)
- **The audit log** is what compliance, legal, and your future self want on day one, not after the incident.

The agent loop is ~40 lines. The trust layer around it is the product.


---

## 🎨 Part 6: Build Your Own Tool! (Challenge)

Now it's your turn. Create a **custom tool** that solves a real problem.

### Idea Starters:
- 🌦️ **Weather lookup** — get weather for a city (simulated)
- 📅 **Meeting scheduler** — check available time slots
- 📧 **Email drafter** — generate email drafts with specific formatting
- 🍴 **Calorie counter** — look up nutrition info for foods
- 📚 **Course catalog** — search Suffolk courses by topic
- 💰 **Tip calculator** — split bills among friends

### Template — Copy and modify:

In [ ]:
# ============================================
# 🎨 YOUR CUSTOM TOOL — Fill in the blanks!
# ============================================

# Step 1: Define your tool (the "job description" for Gemini)
my_custom_tool = types.FunctionDeclaration(
    name="your_tool_name",           # <-- Change this
    description=(
        "Describe what your tool does. Be detailed! "  # <-- Change this
        "Tell Gemini when to use it and what it returns."
    ),
    parameters_json_schema={
        "type": "object",
        "properties": {
            "param1": {                    # <-- Change parameter name
                "type": "string",           # <-- string, number, boolean, etc.
                "description": "What this parameter means"  # <-- Change this
            }
            # Add more parameters here if needed
        },
        "required": ["param1"]             # <-- Which params are required?
    }
)

# Step 2: Write the actual function
def your_tool_name(param1: str) -> str:    # <-- Match the tool name!
    """
    Your tool logic here.
    Must return a string.
    """
    # Your code here!
    return f"Result for {param1}"


# Step 3: Register it with all the other tools
my_tools = [calculator_tool, stock_tool, company_tool, my_custom_tool]
my_functions = {
    "calculator": calculator,
    "get_stock_price": get_stock_price,
    "get_company_info": get_company_info,
    "your_tool_name": your_tool_name        # <-- Match the tool name!
}

# Step 4: Test it!
run_agent_v2(
    "Your test question that would trigger the tool",   # <-- Change this
    my_tools, my_functions
)

---

## 💭 Part 7: Reflection — The Bigger Picture

### What You Just Built vs. What's in Production

| What You Built | What NanoClaw/OpenClaw Do |
|---|---|
| 3 simulated tools | Hundreds of real tools (email, calendar, web, files) |
| Single conversation | Persistent memory across sessions |
| Runs in Colab | Runs in isolated containers 24/7 |
| You type messages | Connected to WhatsApp, Slack, Discord |
| ~50 lines of agent code + a trust layer | NanoClaw: ~500 lines. OpenClaw: ~400,000 lines |

But the **core pattern is identical**:

```
1. Receive message
2. Send to LLM with tool definitions
3. If LLM wants to use a tool → execute it → send result back
4. Repeat until LLM has a final answer
5. Return answer to user
```

**You now understand the architecture of every AI agent in the world.** The complexity in production comes from:
- More tools (and real APIs instead of simulated data)
- Security (containers, permissions, sandboxing)
- Memory (databases, conversation history)
- Reliability (error handling, retries, timeouts)
- UX (WhatsApp integration, web interfaces)

### 💡 The Business Insight

The **most valuable skill** isn't coding the agent loop (that's ~50 lines). It's:
1. **Identifying which tools to build** — What data sources matter? What actions are valuable?
2. **Writing great tool descriptions** — The AI is only as good as its understanding of what each tool does
3. **Designing the user experience** — How do people interact with your agent? When should it ask for help?

These are business decisions, not engineering decisions. That's why **you** are perfectly positioned to build the next generation of AI agents.

---

## 🚀 What's Next?

- **Monday (Session 4, Meeting 2)**: Ship It — you build and deploy a real AI product in one class. Bring a problem worth solving and a teammate.
- **Session 5 (Wed Oct 7)**: Agents Meet the Real World — real APIs, error handling, memory
- **Homework #5 (due Thu Oct 8, 11:59 PM)**: add a 4th tool that solves a real problem you care about, test it with 3 queries (one multi-tool), and answer the three reflection questions. Full brief: course site → Session 4 → Homework #5.

---

**ISOM 260: AI for Business** | Suffolk University | Session 4, Meeting 1

🌐 [isom-260.vercel.app](https://isom-260.vercel.app)